# Introduction

Now you are ready to get a deeper understanding of your data.

Run the following cell to load your data.

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 150

country_pool = ['Italy', 'Portugal', 'US', 'Spain', 'Germany',
                'France', 'Australia', 'New Zealand', 'Canada', 'Chile']
countries = list(np.random.choice(country_pool, n))
countries[0] = 'Italy'
for i in [10, 25, 40]:
    countries[i] = 'Canada'

variety_pool = ['White Blend', 'Portuguese Red', 'Pinot Gris', 'Riesling',
                'Sangiovese', 'Pinot Noir', 'Cabernet Sauvignon', 'Chardonnay',
                'Tempranillo-Merlot', 'Shiraz']
varieties = list(np.random.choice(variety_pool, n))

points = list(np.random.randint(82, 98, n))
prices = [round(float(p), 1) for p in np.random.uniform(10, 200, n)]
prices[5] = 5.0
points[5] = 94

province_map = {
    'Italy': 'Sicily & Sardinia', 'Portugal': 'Douro', 'US': 'California',
    'Spain': 'Northern Spain', 'Germany': 'Rheinhessen', 'France': 'Bordeaux',
    'Australia': 'Victoria', 'New Zealand': "Hawke's Bay",
    'Canada': 'British Columbia', 'Chile': 'Maipo Valley',
}

descriptions = []
for i in range(n):
    if i % 7 == 0:
        descriptions.append('tropical fruit aromas with a fruity finish.')
    elif i % 7 == 1:
        descriptions.append('notes of fruity berries and fresh acidity.')
    elif i % 7 == 2:
        descriptions.append('tropical notes with citrus and dried herbs.')
    else:
        descriptions.append(f'wine description for review {i}.')

reviews = pd.DataFrame({
    'country': countries,
    'description': descriptions,
    'designation': [None if i % 5 == 0 else f'Label {i % 15}' for i in range(n)],
    'points': points,
    'price': prices,
    'province': [province_map[c] for c in countries],
    'region_1': [None if i % 3 == 0 else f'Region {i % 8}' for i in range(n)],
    'region_2': [None if i % 4 == 0 else f'Sub-region {i % 5}' for i in range(n)],
    'taster_name': [None if i % 5 == 0 else f'Taster {i % 6}' for i in range(n)],
    'taster_twitter_handle': [None if i % 5 == 0 else f'@taster{i % 6}' for i in range(n)],
    'title': [f'Winery {i % 20} {2010 + i % 10} Vintage Wine' for i in range(n)],
    'variety': varieties,
    'winery': [f'Winery {i % 20}' for i in range(n)],
})

pd.set_option('display.max_rows', 5)
print(f'Loaded {len(reviews)} wine reviews with {reviews.shape[1]} columns.')
print('Setup complete.')
reviews.head()

Loaded 150 wine reviews with 13 columns.
Setup complete.


,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,Italy,tropical fruit aromas with a fruity finish.,NaN,86,55.6,Sicily & Sardinia,NaN,NaN,NaN,NaN,Winery 0 2010 Vintage Wine,Pinot Gris,Winery 0
1,Spain,notes of fruity berries and fresh acidity.,Label 1,89,27.8,Northern Spain,Region 1,Sub-region 1,Taster 1,@taster1,Winery 1 2011 Vintage Wine,Cabernet Sauvignon,Winery 1
2,New Zealand,tropical notes with citrus and dried herbs.,Label 2,94,44.7,Hawke's Bay,Region 2,Sub-region 2,Taster 2,@taster2,Winery 2 2012 Vintage Wine,Pinot Noir,Winery 2
3,Germany,wine description for review 3.,Label 3,82,187.6,Rheinhessen,NaN,Sub-region 3,Taster 3,@taster3,Winery 3 2013 Vintage Wine,Chardonnay,Winery 3
4,Australia,wine description for review 4.,Label 4,95,131.3,Victoria,Region 4,NaN,Taster 4,@taster4,Winery 4 2014 Vintage Wine,Tempranillo-Merlot,Winery 4


# Exercises

## 1.

What is the median of the `points` column in the `reviews` DataFrame?

In [2]:
median_points = reviews.points.median()
median_points

np.float64(90.0)

**Key insight:** `.median()` is more robust to outliers than `.mean()`. Other useful single-column summary statistics: `.mean()`, `.std()`, `.min()`, `.max()`, `.describe()` (all at once).

## 2.

What countries are represented in the dataset? (Your answer should not include any duplicates.)

In [3]:
countries = reviews.country.unique()
print(countries)

<StringArray>
[      'Italy',       'Spain', 'New Zealand',     'Germany',   'Australia',
       'Chile',          'US',      'Canada',      'France',    'Portugal']
Length: 10, dtype: str


**Key insight:** `.unique()` returns a **numpy array** (not a pandas Series) of unique values in the order they first appear. For a sorted list: `sorted(reviews.country.unique())` or `reviews.country.drop_duplicates()`.

## 3.

How often does each country appear in the dataset? Create a Series `reviews_per_country` mapping countries to the count of reviews of wines from that country.

In [4]:
reviews_per_country = reviews.country.value_counts()
reviews_per_country

country
Australia    20
Canada       20
             ..
Portugal     12
France        9
Name: count, Length: 10, dtype: int64

**Key insight:** `.value_counts()` returns a Series sorted by **count descending** by default — the most common value is at the top. The index holds the unique values; the values hold the counts.

## 4.

Create variable `centered_price` containing a version of the `price` column with the mean price subtracted.

(Note: this 'centering' transformation is a common preprocessing step before applying various machine learning algorithms.)

In [5]:
centered_price = reviews.price - reviews.price.mean()
centered_price

0     -52.190667
1     -79.990667
         ...    
148     2.609333
149   -14.890667
Name: price, Length: 150, dtype: float64

**Alternative using `.map()`:**
```python
centered_price = reviews.price.map(lambda p: p - reviews.price.mean())
```

**Key insight:** Direct vectorized arithmetic (`series - scalar`) is faster than `.map()` because it operates in NumPy's C layer. Prefer vectorized operations over `.map()` whenever the calculation can be expressed element-wise.

## 5.

I'm an economical wine buyer. Which wine is the "best bargain"? Create a variable `bargain_wine` with the title of the wine with the highest points-to-price ratio in the dataset.

In [6]:
highest_ratio = (reviews.points / reviews.price).idxmax()
bargain_wine = reviews.loc[highest_ratio, 'title']
print(f'Best bargain index: {highest_ratio}')
print(f'Wine: {bargain_wine}')

Best bargain index: 5
Wine: Winery 5 2015 Vintage Wine


**Key insight:** `.idxmax()` returns the **index label** of the maximum value (not the value itself). Use it for argmax-style lookups — find where the max lives, then fetch related data with `.loc[]`.

- `.max()` → returns the maximum value
- `.idxmax()` → returns the index label of that maximum value

## 6.

There are only so many words you can use when describing a bottle of wine. Is a wine more likely to be "tropical" or "fruity"? Create a Series `descriptor_counts` counting how many times each of these two words appears in the `description` column in the dataset. (For simplicity, let's ignore the capitalized versions of these words.)

In [7]:
tropical_count = reviews.description.map(lambda d: 'tropical' in d).sum()
fruity_count = reviews.description.map(lambda d: 'fruity' in d).sum()
descriptor_counts = pd.Series([tropical_count, fruity_count], index=['tropical', 'fruity'])
descriptor_counts

tropical    44
fruity      44
dtype: int64

**More idiomatic alternative using `.str.contains()`:**
```python
tropical_count = reviews.description.str.contains('tropical').sum()
fruity_count = reviews.description.str.contains('fruity').sum()
descriptor_counts = pd.Series([tropical_count, fruity_count], index=['tropical', 'fruity'])
```

**Key insight:** `series.map(lambda x: 'word' in x)` produces a boolean Series; `.sum()` counts the True values (True = 1, False = 0). The `.str.contains()` accessor is the pandas-native approach and handles NaN values gracefully (returns NaN instead of raising TypeError).

## 7.

We'd like to host these wine reviews on our website, but a rating system ranging from 80 to 100 points is too hard to understand — we'd like to translate them into simple star ratings. A score of 95 or higher counts as 3 stars, a score of at least 85 but less than 95 is 2 stars. Any other score is 1 star.

Also, the Canadian Vintners Association bought a lot of ads on the site, so any wines from Canada should automatically get 3 stars, regardless of points.

Create a series `star_ratings` with the number of stars corresponding to each review in the dataset.

In [8]:
def star(row):
    if row.country == 'Canada':
        return 3
    elif row.points >= 95:
        return 3
    elif row.points >= 85:
        return 2
    else:
        return 1

star_ratings = reviews.apply(star, axis='columns')
print(star_ratings.value_counts().sort_index())
star_ratings

1    23
2    79
3    48
Name: count, dtype: int64


0      2
1      2
      ..
148    3
149    3
Length: 150, dtype: int64

**Key insight:** `df.apply(func, axis='columns')` calls `func` once per **row**, passing the entire row as a Series. Use `axis='columns'` (or `axis=1`) when your function needs values from multiple columns simultaneously.

Compare: `series.map(func)` applies a function element-by-element to a **single column** only.

---
## Session Summary — Day 3 Pandas

| Exercise | Topic | Key Concept |
|----------|-------|-------------|
| 1 | Median of a column | `.median()` — robust to outliers vs `.mean()` |
| 2 | Unique values | `.unique()` → numpy array in first-seen order |
| 3 | Value counts | `.value_counts()` → Series sorted by count descending |
| 4 | Mean centering | `series - scalar` — vectorized arithmetic, faster than `.map()` |
| 5 | Best bargain (argmax) | `.idxmax()` → index label of max; pair with `.loc[]` to fetch related data |
| 6 | Word count in text | `.map(lambda x: 'word' in x).sum()` or `.str.contains('word').sum()` |
| 7 | Row-wise apply | `df.apply(func, axis='columns')` — needed when function uses multiple columns |